# Piezometric Network of Catalonia — Groundwater Analysis

**Author:** Carlos Daniel Muñoz Sánchez  
**Data Source:** Catalan Water Agency (ACA) — Open Data  
**Dataset:** Groundwater piezometric level in Catalonia  
**URL:** https://analisi.transparenciacatalunya.cat/api/views/6899-hrme/rows.csv  
**License:** Open Data Generalitat de Catalunya  

---

## Objectives

1. **Download** historical piezometric data from ACA's monitoring network directly from their public API.
2. **Explore and Clean** the dataset — data quality, temporal, and spatial coverage.
3. **Analyze Temporal Trends** — are groundwater levels in Catalonia rising or falling?
4. **Spatial Visualization** — interactive Folium map showing the piezometer network and their trends.
5. **Identify Patterns** — comparison by water body and seasonal variability.

---

## Hydrogeological Context

Catalonia has a piezometric monitoring network managed by the ACA that records groundwater levels in dozens of wells distributed across various groundwater bodies in Catalonia's internal basins. This monitoring is essential to:

- Detect aquifer overexploitation situations
- Evaluate the impact of droughts on groundwater resources
- Plan sustainable extractions
- Monitor aquifer recovery after restriction periods

The piezometric level (expressed in meters above sea level) is the main indicator of an aquifer's quantitative status.

In [ ]:
# ── LIBRARIES ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import linregress
import folium
from folium.plugins import MarkerCluster
import os
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.labelsize': 11,
})

print('✓ Libraries loaded successfully')

# Ensure the output directory exists within the current project path
script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
output_dir = os.path.join(script_dir, 'output')
os.makedirs(output_dir, exist_ok=True)

---
## 1. Data Download from ACA API

In [ ]:
# ── DIRECT DOWNLOAD FROM ACA PUBLIC API ───────────────────────────────────────
URL_ACA = (
    'https://analisi.transparenciacatalunya.cat'
    '/api/views/6899-hrme/rows.csv?accessType=DOWNLOAD'
)

print('Downloading piezometric network data from ACA...')
try:
    df_raw = pd.read_csv(URL_ACA)
    print(f'✓ Data downloaded successfully ({len(df_raw):,} rows)')
except Exception as e:
    print(f'Download error: {e}')
    print('Check your internet connection and try again.')

---
## 2. Exploration and Data Cleaning

In [ ]:
# ── INITIAL EXPLORATION ────────────────────────────────────────────────────────
print('GENERAL DATASET INFORMATION')
print('=' * 50)
print(f'Period:           {df_raw.iloc[:,0].min()} → {df_raw.iloc[:,0].max()}')
print(f'Total records:    {len(df_raw):,}')
print(f'\nData types:')
print(df_raw.dtypes)
print(f'\nNull values per column:')
print(df_raw.isnull().sum())

In [ ]:
# ── CLEANING AND STANDARDIZATION ───────────────────────────────────────────────
df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

col_map = {}
for col in df.columns:
    if 'data' in col or 'fecha' in col or 'date' in col: col_map['fecha'] = col
    elif 'estaci' in col or 'estacion' in col or 'station' in col: col_map['estacion'] = col
    elif 'massa' in col or 'masa' in col or 'mass' in col: col_map['masa_agua'] = col
    elif 'utm_x' in col or 'coord_x' in col or 'x_utm' in col: col_map['x'] = col
    elif 'utm_y' in col or 'coord_y' in col or 'y_utm' in col: col_map['y'] = col
    elif 'nivell' in col or 'nivel' in col or 'level' in col or 'alcada' in col: col_map['nivel'] = col

df = df.rename(columns={v: k for k, v in col_map.items()})

df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')
df['nivel'] = pd.to_numeric(df['nivel'], errors='coerce')
df['x'] = pd.to_numeric(df['x'], errors='coerce')
df['y'] = pd.to_numeric(df['y'], errors='coerce')

df = df.dropna(subset=['fecha', 'nivel', 'x', 'y'])

# Logical physical filter for Catalonia
df = df[(df['nivel'] >= -50) & (df['nivel'] <= 3000)]

df['año'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month

print(f'✓ Clean and filtered dataset: {len(df):,} valid records')

---
## 3. Exploratory Analysis

In [ ]:
# ── TEMPORAL AND SPATIAL DISTRIBUTION ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

registros_año = df.groupby('año').size()
axes[0].bar(registros_año.index, registros_año.values, color='#2a9d8f', alpha=0.85)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of measurements')
axes[0].set_title('Records per year', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

axes[1].hist(df['nivel'], bins=50, color='#457b9d', alpha=0.85, edgecolor='white')
axes[1].axvline(df['nivel'].median(), color='#e63946', lw=2, ls='--',
                label=f'Median: {df["nivel"].median():.1f} m')
axes[1].set_xlabel('Piezometric level (m a.s.l.)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of levels', fontweight='bold')
axes[1].legend()

top_masas = df['masa_agua'].value_counts().head(10)
axes[2].barh(range(len(top_masas)), top_masas.values, color='#e9c46a', alpha=0.85)
axes[2].set_yticks(range(len(top_masas)))
axes[2].set_yticklabels([m[:30] + '...' if len(m) > 30 else m for m in top_masas.index], fontsize=8)
axes[2].invert_yaxis()
axes[2].set_xlabel('Number of records')
axes[2].set_title('Top 10 water bodies', fontweight='bold')

fig.suptitle('Exploratory Analysis — ACA Piezometric Network', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, '01_exploratorio.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure saved: output/01_exploratorio.png')

---
## 4. Temporal Trend Analysis

In [ ]:
# ── OVERALL NETWORK TREND ──────────────────────────────────────────────────────
tendencia_mensual = (
    df.groupby(df['fecha'].dt.to_period('M'))['nivel']
    .agg(['mean', 'median', 'std', 'count'])
    .reset_index()
)
tendencia_mensual['fecha'] = tendencia_mensual['fecha'].dt.to_timestamp()
tendencia_mensual = tendencia_mensual[tendencia_mensual['count'] >= 5]

x_num = (tendencia_mensual['fecha'] - tendencia_mensual['fecha'].min()).dt.days
slope, intercept, r_val, p_val, _ = linregress(x_num, tendencia_mensual['median'])
tendencia_lineal = slope * x_num + intercept
tendencia_anual = slope * 365

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(
    tendencia_mensual['fecha'],
    tendencia_mensual['median'] - tendencia_mensual['std'],
    tendencia_mensual['median'] + tendencia_mensual['std'],
    alpha=0.2, color='#457b9d', label='±1 Std. Dev.'
)
ax.plot(tendencia_mensual['fecha'], tendencia_mensual['median'],
        color='#2a9d8f', lw=1.5, ls='--', label='Monthly network median')

color_tend = '#e63946' if tendencia_anual < 0 else '#2a9d8f'
ax.plot(tendencia_mensual['fecha'], tendencia_lineal,
        color=color_tend, lw=2.5,
        label=f'Trend: {tendencia_anual:+.3f} m/year (R²={r_val**2:.2f}, p={p_val:.3f})')

ax.set_xlabel('Date')
ax.set_ylabel('Piezometric level (m a.s.l.)')
ax.set_title('Temporal Evolution of Piezometric Levels — ACA Network Catalonia', fontweight='bold')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, '02_tendencia_general.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure saved: output/02_tendencia_general.png')
print(f'\nOverall network trend: {tendencia_anual:+.4f} m/year')
print(f'p-value: {p_val:.4f} → {"statistically significant" if p_val < 0.05 else "not significant"}')

In [ ]:
# ── TREND BY INDIVIDUAL STATION ───────────────────────────────────────────────
def calcular_tendencia(grupo):
    grupo = grupo.sort_values('fecha').dropna(subset=['nivel'])
    if len(grupo) < 10: return None
    x = (grupo['fecha'] - grupo['fecha'].min()).dt.days
    y = grupo['nivel']
    slope, intercept, r_val, p_val, _ = linregress(x, y)
    return pd.Series({
        'tendencia_m_año': slope * 365,
        'r2': r_val ** 2,
        'p_valor': p_val,
        'n_mediciones': len(grupo),
        'nivel_medio': grupo['nivel'].mean(),
        'rango_m': grupo['nivel'].max() - grupo['nivel'].min(),
        'año_inicio': grupo['fecha'].min().year,
        'año_fin': grupo['fecha'].max().year,
        'x': grupo['x'].iloc[0],
        'y': grupo['y'].iloc[0],
        'masa_agua': grupo['masa_agua'].iloc[0],
    })

tendencias = df.groupby('estacion').apply(calcular_tendencia).dropna().reset_index()
tendencias.to_csv(os.path.join(output_dir, 'resumen_tendencias_estaciones.csv'), index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].hist(tendencias['tendencia_m_año'], bins=60, range=(-5, 5), color='#2a9d8f', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='black', lw=1.5, ls='--', label='No trend')
axes[0].set_xlim(-5, 5)
axes[0].set_xlabel('Trend (m/year)')
axes[0].set_ylabel('Number of stations')
axes[0].set_title('Distribution of trends by station (Zoom)', fontweight='bold')
axes[0].legend()

tend_masa = tendencias.groupby('masa_agua')['tendencia_m_año'].agg(['mean', 'count'])
tend_masa = tend_masa[tend_masa['count'] >= 2]
top_drops = tend_masa.sort_values('mean').head(10)
top_rises = tend_masa.sort_values('mean').tail(10)
extremos_masa = pd.concat([top_drops, top_rises]).sort_values('mean')

colors_bar = ['#e63946' if v < 0 else '#2a9d8f' for v in extremos_masa['mean']]
axes[1].barh(range(len(extremos_masa)), extremos_masa['mean'], color=colors_bar, alpha=0.85)
axes[1].set_yticks(range(len(extremos_masa)))
axes[1].set_yticklabels([f"{m[:30]} ({n})" for m, n in zip(extremos_masa.index, extremos_masa['count'])], fontsize=8)
axes[1].axvline(0, color='black', lw=1, ls='--')
axes[1].set_xlabel('Mean trend (m/year)')
axes[1].set_title('Top 10 Drops and Top 10 Rises by Water Body', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, '03_distribucion_tendencias.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure saved: output/03_distribucion_tendencias.png')

In [ ]:
# ── TIME SERIES FOR MAIN STATIONS ─────────────────────────────────────────────
top_estaciones = df.groupby('estacion').size().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
palette = ['#e63946', '#2a9d8f', '#457b9d', '#e9c46a', '#f4a261', '#264653']

for i, estacion in enumerate(top_estaciones):
    datos_est = df[df['estacion'] == estacion].sort_values('fecha')
    masa = datos_est['masa_agua'].iloc[0]
    
    datos_est = datos_est.set_index('fecha')
    suavizado = datos_est['nivel'].rolling(window=3, center=True).mean()
    
    x_num = (datos_est.index - datos_est.index.min()).days
    slope, intercept, r_val, _, _ = linregress(x_num, datos_est['nivel'])
    tend_anual = slope * 365
    tend_linea = slope * x_num + intercept

    color = palette[i]
    axes[i].plot(datos_est.index, datos_est['nivel'], 'o', ms=2, alpha=0.4, color=color)
    axes[i].plot(datos_est.index, suavizado, '-', lw=2, color=color, label='3M Moving average')
    axes[i].plot(datos_est.index, tend_linea, '--k', lw=1.5, label=f'Trend: {tend_anual:+.3f} m/year')

    axes[i].set_title(f'{estacion[:30]}\n{masa[:30]}', fontsize=9, fontweight='bold')
    axes[i].set_ylabel('Level (m a.s.l.)')
    axes[i].legend(fontsize=7)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=45)

fig.suptitle('Time Series — Stations with Highest Historical Coverage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, '04_series_temporales.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure saved: output/04_series_temporales.png')

In [ ]:
# ── SEASONAL VARIABILITY ───────────────────────────────────────────────────────
df_estacional = df.copy()
media_estacion = df_estacional.groupby('estacion')['nivel'].transform('mean')
df_estacional['anomalia'] = df_estacional['nivel'] - media_estacion
estacional = df_estacional.groupby('mes')['anomalia'].agg(['mean', 'std'])
meses = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(12, 5))
colors_mes = ['#2a9d8f' if v >= 0 else '#e63946' for v in estacional['mean']]
ax.bar(range(1, 13), estacional['mean'], color=colors_mes, alpha=0.85, yerr=estacional['std'] / 2, capsize=4)
ax.axhline(0, color='black', lw=1, ls='--')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses)
ax.set_ylabel('Mean piezometric anomaly (m)')
ax.set_title('Seasonal Variability — Annual Cycle of Piezometric Levels', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, '05_estacionalidad.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure saved: output/05_estacionalidad.png')

---
## 5. Interactive Folium Map

In [ ]:
# ── INTERACTIVE MAP WITH CLUSTERS AND LAYERS ───────────────────────────────────
try:
    import pyproj
    transformer = pyproj.Transformer.from_crs('EPSG:25831', 'EPSG:4326', always_xy=True)
    lon, lat = transformer.transform(tendencias['x'].values, tendencias['y'].values)
    tendencias['lat'], tendencias['lon'] = lat, lon
except ImportError:
    tendencias['lat'] = (tendencias['y'] - 4500000) / 111320 + 40.5
    tendencias['lon'] = (tendencias['x'] - 350000) / (111320 * 0.766) + 2.0

mask = (tendencias['lat'] > 40.0) & (tendencias['lat'] < 43.0) & (tendencias['lon'] > 0.0) & (tendencias['lon'] < 4.0)
tendencias_mapa = tendencias[mask].copy()

mapa = folium.Map(location=[41.8, 1.8], zoom_start=8, tiles='CartoDB positron', name='Light Map')

def tendencia_color(tend):
    if tend < -0.5: return '#d62728'
    elif tend < -0.1: return '#ff7f0e'
    elif tend < 0.1: return '#bcbd22'
    elif tend < 0.5: return '#2ca02c'
    else: return '#1f77b4'

group_negative = folium.FeatureGroup(name='🔴 Negative Trends (Drop)')
cluster_negative = MarkerCluster().add_to(group_negative)

group_positive = folium.FeatureGroup(name='🟢 Positive/Stable Trends (Rise)')
cluster_positive = MarkerCluster().add_to(group_positive)

for _, row in tendencias_mapa.iterrows():
    color = tendencia_color(row['tendencia_m_año'])
    popup_html = f"<div style='font-family: Arial; width: 200px;'><b>{row['estacion']}</b><br>Trend: {row['tendencia_m_año']:+.3f} m/year</div>"
    
    marker = folium.CircleMarker(
        location=[row['lat'], row['lon']], radius=6, color='white', weight=1,
        fill=True, fill_color=color, fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=220)
    )
    
    if row['tendencia_m_año'] < 0:
        marker.add_to(cluster_negative)
    else:
        marker.add_to(cluster_positive)

group_negative.add_to(mapa)
group_positive.add_to(mapa)
folium.LayerControl(collapsed=False).add_to(mapa)

# Add Title to Map
title_html = '''
<div style="position: fixed; top: 15px; left: 50px; width: 380px; height: 45px; 
    background-color: white; border: 2px solid grey; z-index:9999; font-size:15px; 
    font-weight: bold; text-align: center; line-height: 45px; border-radius: 5px; box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
    ACA Piezometric Network - Trends Map
</div>
'''
mapa.get_root().html.add_child(folium.Element(title_html))

# Add Legend to Map
legend_html = '''
<div style="position: fixed; bottom: 30px; right: 30px; width: 220px; height: 160px; 
    background-color: white; border: 2px solid grey; z-index:9999; font-size:12px; 
    padding: 10px; border-radius: 5px; box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
    <b>Piezometric Trend (m/year)</b><br>
    <i style="background:#d62728;width:15px;height:15px;float:left;margin-right:8px;border-radius:50%;"></i> < -0.5 (Severe Drop)<br>
    <i style="background:#ff7f0e;width:15px;height:15px;float:left;margin-right:8px;border-radius:50%;"></i> -0.5 to -0.1 (Moderate Drop)<br>
    <i style="background:#bcbd22;width:15px;height:15px;float:left;margin-right:8px;border-radius:50%;"></i> -0.1 to 0.1 (Stable)<br>
    <i style="background:#2ca02c;width:15px;height:15px;float:left;margin-right:8px;border-radius:50%;"></i> 0.1 to 0.5 (Moderate Rise)<br>
    <i style="background:#1f77b4;width:15px;height:15px;float:left;margin-right:8px;border-radius:50%;"></i> > 0.5 (Significant Rise)<br>
</div>
'''
mapa.get_root().html.add_child(folium.Element(legend_html))

mapa.save(os.path.join(output_dir, 'mapa_piezometrico_catalunya.html'))
print('✓ Interactive clustered & layered map successfully saved.')

---
## 6. Summary and Conclusions

In [ ]:
# ── FINAL SUMMARY TABLE ────────────────────────────────────────────────────────
resumen_tabla = pd.DataFrame({
    'Metric': [
        'Total Raw Records',
        'Clean Valid Records',
        'Total Piezometers Analyzed',
        'Network Median Level (m a.s.l.)',
        'Overall Annual Trend (m/year)',
        'Stations with Negative Trend',
        'Stations with Positive Trend'
    ],
    'Value': [
        f"{len(df_raw):,}",
        f"{len(df):,}",
        f"{len(tendencias):,}",
        f"{df['nivel'].median():.2f} m",
        f"{tendencia_anual:+.3f} m/year",
        f"{(tendencias['tendencia_m_año'] < 0).sum()} ({((tendencias['tendencia_m_año'] < 0).mean() * 100):.1f%})",
        f"{(tendencias['tendencia_m_año'] >= 0).sum()} ({((tendencias['tendencia_m_año'] >= 0).mean() * 100):.1f%})"
    ]
})

print('\n' + '='*50)
print(' ACA PIEZOMETRIC ANALYSIS SUMMARY')
print('='*50)
print(resumen_tabla.to_string(index=False))
print('='*50)
print(f"All generated files have been saved in: {output_dir}")

---
## 7. Discussion and Interpretation

### Quantitative Overview and Network Scale
The analysis of the ACA monitoring system across **907 total piezometers analyzed** from a robust raw volume of **243,969 records** (**238,186 valid cleaned records**) provides a comprehensive macro-scale view of Catalonia's underground water storage. With a network median level situated at **21.40 m above sea level**, the baseline distribution captures the transition from high-altitude inland recharge zones to low-lying coastal alluvial discharge systems.

### Spatial Patterns of Piezometric Trends
The spatial distribution of trends mapped through the interactive Folium viewer reveals a fine balance plagued by deep local polarizations. Despite an overall network-wide aggregate trend showing an apparent positive slope of **+4.816 m/year** (heavily weighted by localized recoveries in heavily monitored or artificially recharged basins), the station-by-station breakdown tells a more critical story: **426 stations (47.0%) exhibit a negative trend**, while **481 stations (53.0%) show a positive trend**. This nearly 50/50 split emphasizes that while recovery or stabilization occurs in over half the network, nearly half of Catalonia's monitored points remain under severe quantitative pressure ($<-0.5\text{ m/year}$), pointing toward localized structural overexploitation or vulnerability to prolonged meteorological droughts.

### Hydrogeological Inertia and Climate Drivers
Groundwater levels act as a smoothed, integrated response to climatic variability. Unlike surface water bodies or reservoir storage—which react rapidly to episodic intense rainfall—piezometric levels display a marked hydrogeological inertia. The downward trajectories captured in nearly half of the stations reflect cumulative deficits in natural recharge, exacerbated by structural pumping exceeding the sustainable yield of those specific aquifers.

### Seasonality and Management Implications
The seasonal anomaly composite highlights a robust cyclical pattern characterized by negative anomalies during late summer and early autumn (driven by peak irrigation extraction and high evapotranspiration rates) and a subsequent rebound towards positive anomalies in late winter/spring. Understanding this exact intra-annual window is vital for the Catalan Water Agency (ACA) to optimize monitoring frequencies, enforce dynamic abstraction controls, and plan non-conventional water integration (such as reclaimed water or desalination) before critical depletion thresholds are breached.

---

## References

* Agència Catalana de l'Aigua (ACA). (2024). *Nivell piezomètric de les aigües subterrànies de Catalunya*. Generalitat de Catalunya — Open Data.
* Agència Catalana de l'Aigua (ACA). (2020). *Pla especial d'actuació en situació d'alerta i eventual sequera (PES)*. Generalitat de Catalunya.